In [2]:
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords

# ML
from sklearn.model_selection import train_test_split

In [3]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\subra\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
df = pd.read_csv("Twitter_Data.csv")

In [5]:
df.head()

,clean_text,category
0,when modi promised “minimum government maximum...,-1.0
1,talk all the nonsense and continue all the dra...,0.0
2,what did just say vote for modi welcome bjp t...,1.0
3,asking his supporters prefix chowkidar their n...,1.0
4,answer who among these the most powerful world...,1.0


In [6]:
df.columns


Index(['clean_text', 'category'], dtype='object')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 162980 entries, 0 to 162979
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   clean_text  162976 non-null  object 
 1   category    162973 non-null  float64
dtypes: float64(1), object(1)
memory usage: 2.5+ MB


In [8]:
df['category'].value_counts()

category
 1.0    72250
 0.0    55213
-1.0    35510
Name: count, dtype: int64

In [9]:
df['sentiment'] = df['category'].map({
    1: 'Positive',
    0: 'Neutral',
    -1: 'Negative'
})

In [10]:
df[['clean_text', 'sentiment']].head()

,clean_text,sentiment
0,when modi promised “minimum government maximum...,Negative
1,talk all the nonsense and continue all the dra...,Neutral
2,what did just say vote for modi welcome bjp t...,Positive
3,asking his supporters prefix chowkidar their n...,Positive
4,answer who among these the most powerful world...,Positive


## NLP Preprocessing

In [11]:
stop_words = set(stopwords.words('english'))

In [12]:
df['clean_text'] = df['clean_text'].fillna("")

In [13]:
def preprocess_text(text):
    
    # 1. Convert to lowercase
    text = text.lower()
    
    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # 3. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # 4. Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # 5. Tokenization
    words = text.split()
    
    # 6. Remove stopwords
    words = [word for word in words if word not in stop_words]
    
    # 7. Join words back
    text = " ".join(words)
    
    return text

In [18]:
df['processed_text'] = df['clean_text'].apply(preprocess_text)

In [19]:
df[['clean_text', 'processed_text']].head()

,clean_text,processed_text
0,when modi promised “minimum government maximum...,modi promised “minimum government maximum gove...
1,talk all the nonsense and continue all the dra...,talk nonsense continue drama vote modi
2,what did just say vote for modi welcome bjp t...,say vote modi welcome bjp told rahul main camp...
3,asking his supporters prefix chowkidar their n...,asking supporters prefix chowkidar names modi ...
4,answer who among these the most powerful world...,answer among powerful world leader today trump...


In [20]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    
    words = text.split()
    words = [word for word in words if word not in stop_words]
    
    # Stemming
    words = [stemmer.stem(word) for word in words]
    
    return " ".join(words)

## STEP 3: Feature Engineering

In [21]:
X = df['processed_text']
y = df['sentiment']

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### PART A: Bag of Words (BoW)

In [23]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

In [24]:
X_train_bow.shape

(130384, 5000)

### PART B: TF-IDF

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [26]:

X_train_tfidf.shape

(130384, 5000)

# STEP 4: Model Building (Logistic, Naive Bayes, Decision Tree)

### 1. Logistic Regression

In [27]:
df = df.dropna(subset=['processed_text', 'sentiment'])

In [28]:
X = df['processed_text']
y = df['sentiment']

In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [33]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

In [36]:
from sklearn.linear_model import LogisticRegression

In [37]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_bow, y_train)

y_pred_lr = lr.predict(X_test_bow)

### 2. Naive Bayes

In [38]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train_bow, y_train)

y_pred_nb = nb.predict(X_test_bow)

### 3. Decision Tree

In [40]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train_bow, y_train)

y_pred_dt = dt.predict(X_test_bow)

### PART B: Using TF-IDF

In [42]:
df = df.dropna(subset=['processed_text', 'sentiment'])
df = df.reset_index(drop=True)

In [43]:
X = df['processed_text']
y = df['sentiment']

In [44]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [46]:
from sklearn.linear_model import LogisticRegression

lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_tfidf, y_train)

y_pred_lr_tfidf = lr_tfidf.predict(X_test_tfidf)

In [48]:
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)

y_pred_nb_tfidf = nb_tfidf.predict(X_test_tfidf)

In [49]:
dt_tfidf = DecisionTreeClassifier()
dt_tfidf.fit(X_train_tfidf, y_train)

y_pred_dt_tfidf = dt_tfidf.predict(X_test_tfidf)

In [50]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [51]:
def evaluate_model(y_test, y_pred):
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1 Score": f1_score(y_test, y_pred, average='weighted')
    }

In [52]:
results = {}

results['LR_BoW'] = evaluate_model(y_test, y_pred_lr)
results['NB_BoW'] = evaluate_model(y_test, y_pred_nb)
results['DT_BoW'] = evaluate_model(y_test, y_pred_dt)

In [53]:
results['LR_TFIDF'] = evaluate_model(y_test, y_pred_lr_tfidf)
results['NB_TFIDF'] = evaluate_model(y_test, y_pred_nb_tfidf)
results['DT_TFIDF'] = evaluate_model(y_test, y_pred_dt_tfidf)

In [54]:
results_df = pd.DataFrame(results).T
results_df

,Accuracy,Precision,Recall,F1 Score
LR_BoW,0.898543,0.899123,0.898543,0.897751
NB_BoW,0.783678,0.785659,0.783678,0.783870
DT_BoW,0.843442,0.842652,0.843442,0.842859
LR_TFIDF,0.891364,0.893158,0.891364,0.890115
NB_TFIDF,0.729130,0.767888,0.729130,0.718096
DT_TFIDF,0.805768,0.804053,0.805768,0.804605


📌 Insights
TF-IDF performed better than BoW because it gives importance to meaningful words.
Logistic Regression achieved the best performance among all models.
Naive Bayes performed well with text data due to probability-based approach.
Decision Tree showed lower performance due to overfitting on sparse data.
Preprocessing significantly improved model accuracy.

Raw Text → Cleaning → Preprocessing → Vectorization → Model → Evaluation

In [55]:
print("X_train_tfidf:", X_train_tfidf.shape)
print("y_train:", y_train.shape)

X_train_tfidf: (130378, 5000)
y_train: (130378,)


In [56]:
print("X_test_tfidf:", X_test_tfidf.shape)
print("y_test:", y_test.shape)

X_test_tfidf: (32595, 5000)
y_test: (32595,)


In [57]:
print(df['processed_text'].isnull().sum())
print(df['sentiment'].isnull().sum())

0
0


In [58]:
print(y_train.unique())

['Positive' 'Negative' 'Neutral']


In [59]:
print(y_pred_lr_tfidf[:5])

['Neutral' 'Neutral' 'Neutral' 'Positive' 'Positive']


In [63]:
text = "I really love this product, it's amazing!"
processed = preprocess_text(text)

vector = tfidf.transform([processed])
prediction = lr_tfidf.predict(vector)

print("Predicted Sentiment:", prediction[0])

Predicted Sentiment: Positive


In [67]:
samples = [
    "This is the best thing ever!",
    "Worst service, very disappointed",
    "It is okay, nothing special"
]

for text in samples:
    processed = preprocess_text(text)
    
    vector = tfidf.transform([processed])
    pred = lr_tfidf.predict(vector)
    
    print("\nText:", text)
    print("Processed:", processed)
    print("Prediction:", pred[0])


Text: This is the best thing ever!
Processed: best thing ever
Prediction: Positive

Text: Worst service, very disappointed
Processed: worst servic disappoint
Prediction: Negative

Text: It is okay, nothing special
Processed: okay noth special
Prediction: Positive
